# Module 01 — LLM Basics (Google AI Studio)

**Goal:** Make your first LLM API call from a Databricks notebook and understand every part of it.

This notebook runs on the **Databricks free tier** and talks to a Google Gemini model through the [Google AI Studio](https://aistudio.google.com/) REST API — free with any Google account.

We call the API with plain `requests` (no SDK) so you can see the exact JSON going over the wire. The curl equivalent is:

```bash
curl "https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent" \
  -H 'Content-Type: application/json' \
  -H 'X-goog-api-key: <API token>' \
  -X POST \
  -d '{"contents": [{"parts": [{"text": "Explain how AI works in a few words"}]}]}'
```

By the end you will know:
- What `contents` and `parts` are, and why roles matter
- What the raw API response object looks like
- Why we set `temperature=0` for deterministic tasks like SQL generation
- How to keep a multi-turn conversation

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install `requests`

`requests` is usually already in the Databricks runtime, but we pin it explicitly so this notebook is self-contained. Run once per session, then restart Python so the import is fresh.

In [6]:
%pip install --quiet requests
dbutils.library.restartPython()

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


NameError: name 'dbutils' is not defined

## Step 2 — Set your parameters

We use **Databricks notebook widgets** so you never hard-code secrets. Run the cell below once — three input boxes appear at the top of the notebook.

| Widget | Default | What to do |
|--------|---------|------------|
| `API_KEY` | *(empty)* | Paste your Google AI Studio key — create one at https://aistudio.google.com/apikey |
| `BASE_URL` | `https://generativelanguage.googleapis.com/v1beta` | Leave as-is |
| `MODEL` | `gemini-flash-latest` | Leave as-is, or pick another from https://ai.google.dev/gemini-api/docs/models |

> Want a different Gemini model (e.g. `gemini-2.5-pro`, `gemini-2.5-flash`)? Just change the `MODEL` widget — everything else stays the same.

In [ ]:
dbutils.widgets.text("API_KEY", "", "API key")
dbutils.widgets.text("BASE_URL", "https://generativelanguage.googleapis.com/v1beta", "Base URL")
dbutils.widgets.text("MODEL", "gemini-flash-latest", "Model")

API_KEY = dbutils.widgets.get("API_KEY")
BASE_URL = dbutils.widgets.get("BASE_URL")
MODEL = dbutils.widgets.get("MODEL")

if not API_KEY:
    raise ValueError("Paste your API key into the API_KEY widget at the top of the notebook.")

print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")
print("API key:  set")

NameError: name 'dbutils' is not defined

## Step 3 — Build a tiny client

Google's REST API takes a POST to `{BASE_URL}/models/{MODEL}:generateContent` with the API key in the `X-goog-api-key` header. The whole "client" is a single helper that POSTs JSON and returns the parsed response.

In [ ]:
import requests

def generate_content(body: dict, model: str = MODEL) -> dict:
    url = f"{BASE_URL}/models/{model}:generateContent"
    headers = {
        "Content-Type": "application/json",
        "X-goog-api-key": API_KEY,
    }
    r = requests.post(url, headers=headers, json=body, timeout=60)
    r.raise_for_status()
    return r.json()

## Step 4 — Your first API call

The minimum required field is `contents`. `contents` is a list of conversation turns. Each turn has:
- `role`: who is speaking — `"user"` or `"model"` (Gemini's name for the assistant)
- `parts`: a list of content chunks — for text, each part is `{"text": "..."}`

The `role` is optional for a single-turn request, but we include it explicitly so the shape generalises to multi-turn.

In [ ]:
response = generate_content({
    "contents": [
        {"role": "user", "parts": [{"text": "What is 2 + 2? Answer in one word."}]}
    ]
})

print(response["candidates"][0]["content"]["parts"][0]["text"])

## Step 5 — Inspect the response

There's more in the response than just the text. The fields below show up again in later modules.

In [ ]:
candidate = response["candidates"][0]
usage = response.get("usageMetadata", {})

print(f"model used:      {response.get('modelVersion', MODEL)}")
print(f"finish_reason:   {candidate.get('finishReason')}")
print(f"content:         {candidate['content']['parts'][0]['text']}")
print(f"prompt tokens:   {usage.get('promptTokenCount')}")
print(f"response tokens: {usage.get('candidatesTokenCount')}")
print(f"total tokens:    {usage.get('totalTokenCount')}")

## Step 6 — The system instruction

Gemini has a dedicated `systemInstruction` field that sits **outside** of `contents`. It gives the model persistent instructions that apply to every user message. It's the most powerful knob for controlling model behavior.

In [ ]:
def ask(question: str, system: str = "") -> str:
    body = {
        "contents": [
            {"role": "user", "parts": [{"text": question}]}
        ]
    }
    if system:
        body["systemInstruction"] = {"parts": [{"text": system}]}
    response = generate_content(body)
    return response["candidates"][0]["content"]["parts"][0]["text"]


print("No system instruction:")
print(ask("Tell me about Paris."))
print()
print("With system instruction (one sentence only):")
print(ask(
    "Tell me about Paris.",
    system="You are a tour guide. Answer every question in exactly one sentence.",
))

## Step 7 — Temperature: determinism vs creativity

`temperature=0` always picks the most likely next token → same question, same answer every time.  
`temperature=1` samples from the distribution → same question, different answers each run.

In Gemini's API, sampling knobs live under `generationConfig`.

For SQL generation we always use `temperature=0`. For brainstorming or summarization you'd use 0.7–1.0.

> Free-tier Gemini has rate limits. If you see a 429, wait a few seconds and retry.

In [ ]:
question = "Name one random country."

def ask_with_temperature(temperature: float) -> str:
    response = generate_content({
        "contents": [{"role": "user", "parts": [{"text": question}]}],
        "generationConfig": {"temperature": temperature},
    })
    return response["candidates"][0]["content"]["parts"][0]["text"].strip()


print("temperature=0 (deterministic) — should be the same every run:")
for _ in range(3):
    print(" ", ask_with_temperature(0))

print()
print("temperature=1 (creative) — should vary:")
for _ in range(3):
    print(" ", ask_with_temperature(1))

## Step 8 — Multi-turn conversation

LLMs are **stateless** — they have no memory between API calls. You simulate memory by appending every turn to `contents` before the next call.

Gemini uses `role: "user"` for the human and `role: "model"` for the assistant (where OpenAI would say `assistant`).

In [ ]:
system_instruction = {"parts": [{"text": "You are a helpful assistant."}]}
contents = []

def chat(user_message: str) -> str:
    contents.append({"role": "user", "parts": [{"text": user_message}]})
    response = generate_content({
        "systemInstruction": system_instruction,
        "contents": contents,
    })
    model_message = response["candidates"][0]["content"]["parts"][0]["text"]
    contents.append({"role": "model", "parts": [{"text": model_message}]})
    return model_message


print("Turn 1:", chat("My name is Alice."))
print()
print("Turn 2:", chat("What is my name?"))
print()
print(f"Turns in history: {len(contents)} (grows with every turn!)")

## Summary

| Concept | Key point |
|---------|----------|
| `contents` | The full conversation history sent on every call |
| `systemInstruction` | Persistent instructions that shape every response (top-level field, not a role) |
| `role: "model"` | Gemini's name for assistant turns |
| `finishReason` | Why the model stopped — important in the agentic loop (Module 04) |
| `generationConfig.temperature = 0` | Deterministic output — use this for SQL generation |
| Stateless | LLMs don't remember — you must append history yourself |
| REST | One `POST` + one header (`X-goog-api-key`) is all Google AI Studio needs |

**Next:** Module 02 — how to get reliable *structured* output (JSON) from an LLM instead of free-form text.